# Devign on Colab (free tier)

Runs the real-data Devign reproduction (paper: *Devign: Effective Vulnerability Identification by Learning Comprehensive Program Semantics via Graph Neural Networks*, NeurIPS 2019) end-to-end on a free Colab GPU.

**Before you start:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload `DEVIGEN_src.zip` (the code, no data) to the root of your Google Drive (`My Drive/DEVIGEN_src.zip`). If you'd rather not use Drive, you can instead upload it directly in the cell below via the file picker — see the comment there.

**Free-tier reality check:**
- A free session is capped at ~12h and disconnects on extended idle. This notebook is written to survive that: model checkpoints and metrics go to **Drive** (persistent), while the parsed graph dataset stays on the **local, ephemeral disk** (cheap to rebuild: ~15–25 min on Colab's 2 vCPUs, since re-fetching + re-parsing 27k functions doesn't need to survive a disconnect).
- `scripts.reproduce` is resumable: every stage skips itself if its artifact already exists on Drive. If you get disconnected, just re-run from the top — finished project/model combinations are skipped, not redone.
- Because word2vec is refit from scratch after every reconnect (it's not persisted), model checkpoints trained in different sessions use slightly different (but equally valid) embeddings — a minor source of noise across a multi-session run, not a bug. Set `PERSIST_PROCESSED_TO_DRIVE = True` below if you'd rather pay Drive's slower I/O to avoid that.
- No git remote exists for this repo, so the notebook works from a plain source zip rather than `git clone`.

In [ ]:
# 1. Confirm a GPU is attached (Runtime -> Change runtime type -> T4 GPU if this fails).
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU visible — set Runtime > Change runtime type > T4 GPU, then Runtime > Restart runtime."
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive (for persistent artifacts) and fetch the source zip.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ZIP = '/content/drive/MyDrive/DEVIGEN_src.zip'

if not os.path.exists(DRIVE_ZIP):
    # Fallback: upload the zip directly into this session instead of using Drive.
    print(f"{DRIVE_ZIP} not found — falling back to a direct upload picker.")
    from google.colab import files
    uploaded = files.upload()  # pick DEVIGEN_src.zip from your machine
    DRIVE_ZIP = '/content/' + next(iter(uploaded))

print("Using source zip:", DRIVE_ZIP)

In [ ]:
# 3. Unpack onto the fast LOCAL disk (not Drive — Drive's FUSE mount is much slower for
#    the many small file reads/writes that training and pip do).
!rm -rf /content/DEVIGEN
!mkdir -p /content/DEVIGEN
!unzip -q "{DRIVE_ZIP}" -d /content/DEVIGEN
%cd /content/DEVIGEN
!ls

In [ ]:
# 4. Install dependencies.
#    cppcheck installs via apt here (unlike a plain Windows dev box), so Table 3 gets the REAL
#    tool on both rows instead of the labelled regex fallback.
!apt-get -qq update && apt-get -qq install -y cppcheck
!pip -q install -r requirements.txt
!cppcheck --version
!flawfinder --version

In [ ]:
# 5. Sanity check: the test suite should pass before spending any GPU time.
!python -m pytest tests/ -q

In [ ]:
# 6. Build a Colab-specific config: same hyperparameters, Drive-backed artifacts_dir so
#    checkpoints/metrics survive a disconnect, local-disk processed_dir for speed.
import yaml

PERSIST_PROCESSED_TO_DRIVE = False  # True = slower but processed data also survives disconnects
RUN_NAME = 'run1'                   # change this to start a fresh, independent run

drive_run_dir = f'/content/drive/MyDrive/DEVIGEN_runs/{RUN_NAME}'
os.makedirs(drive_run_dir, exist_ok=True)

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['project']['artifacts_dir'] = f'{drive_run_dir}/artifacts'
if PERSIST_PROCESSED_TO_DRIVE:
    cfg['data']['processed_dir'] = f'{drive_run_dir}/processed'
    cfg['data']['raw_dir'] = f'{drive_run_dir}/raw'
    cfg['data']['hf_cache_dir'] = f'{drive_run_dir}/hf_cache'
else:
    cfg['data']['processed_dir'] = '/content/DEVIGEN/data/processed'
    cfg['data']['raw_dir'] = '/content/DEVIGEN/data/raw'
    cfg['data']['hf_cache_dir'] = '/content/hf_cache'

# Colab's free GPU (T4, 16 GB) has roughly 2x the dev laptop's 8 GB the default was tuned for;
# 12000 is still safe, raise it (e.g. 20000) if you want to trade memory headroom for speed.
# cfg['dataset']['max_nodes_per_batch'] = 20000

with open('config_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print('artifacts_dir  ->', cfg['project']['artifacts_dir'])
print('processed_dir  ->', cfg['data']['processed_dir'])
print('raw_dir        ->', cfg['data']['raw_dir'])

In [ ]:
# 7. Data prep: downloads the Devign authors' released dataset (FFmpeg + QEMU, ~27k real
#    functions) from HuggingFace and builds the composite graphs. ~15-25 min on Colab's vCPUs.
!python -m scripts.prepare_data --config config_colab.yaml

## Run the study

`--epochs` below overrides the paper's 200/patience-100 schedule uniformly across every model in the matrix; lower it if you're working in short free-tier sessions, since the run is fully resumable regardless. **If your runtime disconnects, just re-run the cell below** — `scripts.reproduce` skips every project/model combination whose artifact is already on Drive and only trains what's missing.

In [ ]:
# 8. Train Devign + Ggrn + baselines (per-project + Combined), then Table 3, Q5 holdout,
#    ablation, and the commit-disjoint leakage check. Safe to re-run after a disconnect.
EPOCHS = 60  # paper uses 200/patience-100; lower this for a first pass, raise it for a final run
!python -m scripts.reproduce --config config_colab.yaml --epochs {EPOCHS}

In [ ]:
# 9. Show the result tables.
import json, os

artifacts = cfg['project']['artifacts_dir']
for name in ('table2.json', 'table3.json', 'ablation/ablation.json', 'cve.json', 'leakage_check.json'):
    path = os.path.join(artifacts, name)
    print('=' * 70)
    print(name, ('(missing)' if not os.path.exists(path) else ''))
    if os.path.exists(path):
        with open(path) as f:
            print(json.dumps(json.load(f), indent=2))

## Notes

- All checkpoints, metrics, and the four result JSON files live under `My Drive/DEVIGEN_runs/<RUN_NAME>/artifacts/` — they persist across sessions and disconnects.
- To start a completely independent run (e.g. to compare hyperparameters), change `RUN_NAME` in the config cell before re-running.
- To force a stage to redo instead of being skipped, pass `--force` to `scripts.reproduce` (redoes everything) or delete the specific artifact file/folder under `artifacts/` you want rebuilt.
- `cppcheck` is genuinely installed here via `apt-get`, unlike a plain Windows dev box where it isn't — so on Colab, Table 3 compares Devign against the real tool on both rows, not the labelled heuristic fallback.